In [2]:
!ls -lh /content/news_model.zip

-rw-r--r-- 1 root root 101M Apr 27 08:00 /content/news_model.zip


In [9]:
!unzip -FF /content/news_model.zip -d /content/

Archive:  /content/news_model.zip
  inflating: /content/model.safetensors  
  inflating: /content/tokenizer.json  
  inflating: /content/config.json    
  inflating: /content/tokenizer_config.json  
  inflating: /content/generation_config.json  


In [ ]:
import torch
torch.cuda.is_available()

True

In [24]:
!pip install transformers datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5ccfbeaed836cb804d18b4fc73332f8fd4f384a83dfb8c714200624c64de46e3
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [25]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 98.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [10]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [ ]:
print(dataset["train"][0])

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [13]:
small_train = dataset["train"].shuffle(seed=42).select(range(50000))
small_val = dataset["validation"].shuffle(seed=42).select(range(2000))

In [14]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [15]:
def preprocess(example):
    inputs = ["summarize: " + article for article in example["article"]]
    targets = [h.split("\n")[0] for h in example["highlights"]]

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=64, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [16]:
tokenized_train = small_train.map(preprocess, batched=True)
tokenized_val = small_val.map(preprocess, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=True,
    load_best_model_at_end=True,
    save_total_limit=2
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.536297,0.449243
2,0.497000,0.449213
3,0.490660,0.447710


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=37500, training_loss=0.5230053824869791, metrics={'train_runtime': 4797.2198, 'train_samples_per_second': 31.268, 'train_steps_per_second': 7.817, 'total_flos': 2.03012702208e+16, 'train_loss': 0.5230053824869791, 'epoch': 3.0})

In [20]:
def generate_headline(text):
    inputs = tokenizer("summarize: " + text, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=12,
            num_beams=5,
            length_penalty=2.0,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [22]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [26]:
import evaluate
rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(100):  # you can increase later
    pred = generate_headline(dataset["test"][i]["article"])
    ref = dataset["test"][i]["highlights"].split("\n")[0]

    predictions.append(pred)
    references.append(ref)

results = rouge.compute(predictions=predictions, references=references)
print("ROUGE Scores:", results)

ROUGE Scores: {'rouge1': np.float64(0.24408604142979454), 'rouge2': np.float64(0.1231314239547413), 'rougeL': np.float64(0.22897492383944656), 'rougeLsum': np.float64(0.22942442763646437)}


In [27]:
print("Generated:", generate_headline(dataset["test"][0]["article"]))
print("Actual:", dataset["test"][0]["highlights"])

Generated: The Palestinian Authority officially becomes the 123rd member
Actual: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [28]:
model.save_pretrained("headline-model")
tokenizer.save_pretrained("headline-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('headline-model/tokenizer_config.json', 'headline-model/tokenizer.json')

In [29]:
import os
import shutil
from google.colab import files

# -----------------------------
# 1. Save model + tokenizer
# -----------------------------
save_directory = "news_model"

# Clean old directory if exists (avoids corrupted saves)
if os.path.exists(save_directory):
    shutil.rmtree(save_directory)

model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model saved at: {save_directory}")

# -----------------------------
# 2. Create ZIP file
# -----------------------------
zip_path = shutil.make_archive(save_directory, 'zip', save_directory)

print(f"ZIP created at: {zip_path}")

# -----------------------------
# 3. Download ZIP
# -----------------------------
print("Downloading model...")
files.download(zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: news_model
ZIP created at: /content/news_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>